# Quick run: explainable multi-cue deepfake detection (about 2–3 GPU-hours)

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**.
**Add Input:** `xdxd003/ff-c23`, your private `deepfake-code` dataset, and optionally
`reubensuju/celeb-df-v2` and the competition `deepfake-detection-challenge` (accept its rules first).

Run the first two cells interactively to check that every input is found, then **Save Version → Save & Run All**.
When it finishes, download `download.zip` (models + results) from the Output tab.

Quick-mode settings, which go into Sections 5.1.1, 5.1.5 and 5.2.2 of the report:
16 uniform frames and 4 clips of 16 frames per video; detector on every 4th clip frame with landmarks
interpolated in between; 4 frames per video per training epoch; meta-learner trained on validation-split
predictions (blending) instead of 5-fold out-of-fold predictions; robustness on every 3rd test video at
the strongest level of each perturbation; one seed.

In [ ]:
import glob, os, shutil, subprocess, time
import tensorflow as tf, keras
try:
    import tf_keras
except ImportError:
    subprocess.run(f"pip install -q tf-keras~={'.'.join(tf.__version__.split('.')[:2])}.0", shell=True, check=True)
    import tf_keras
subprocess.run("pip install -q retina-face", shell=True, check=True)
print("Put these versions in requirements.txt on your laptop:")
print(f"  tensorflow=={tf.__version__}\n  keras=={keras.__version__}\n  tf-keras=={tf_keras.__version__}")

In [ ]:
W = "/kaggle/working"; os.chdir(W)

def find(name, isdir=False):
    """Shortest path under /kaggle/input with this file or folder name (follows Kaggle's mount links)."""
    hits = [os.path.join(root, n) for root, dirs, files in os.walk("/kaggle/input", followlinks=True)
            for n in (dirs if isdir else files) if n == name]
    return min(hits, key=len) if hits else None

src = find("prepare_data.py")
if src:
    shutil.copytree(os.path.dirname(src), "code", dirs_exist_ok=True)
else:                                        # Kaggle kept the upload as a zip
    z = find("deepfake-code.zip")
    assert z, "deepfake-code not found: Add Input -> Your Work -> deepfake-code"
    shutil.unpack_archive(z, "code")
FF = os.path.dirname(find("Deepfakes", isdir=True))
CDF = find("List_of_testing_videos.txt"); CDF = CDF and os.path.dirname(CDF)
DFDC = find("train_sample_videos", isdir=True)
N_GPU = max(1, subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.count("GPU "))
print("FF++:", FF, "\nCeleb-DF:", CDF, "\nDFDC:", DFDC, "\nGPUs:", N_GPU)
assert FF, "FF++ input not found: add xdxd003/ff-c23"

os.makedirs("splits", exist_ok=True)
for s in ("train", "val", "test"):
    subprocess.run(f"curl -sSL -o splits/{s}.json https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{s}.json",
                   shell=True, check=True)

Q = "--n-uniform 16 --n-clips 4 --detect-every 4 --max-side 320"
def run(*cmds):
    """Run shell commands in parallel, one per GPU; stop on any failure."""
    t = time.time()
    procs = [subprocess.Popen(c, shell=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i % N_GPU)})
             for i, c in enumerate(cmds)]
    last = t
    while any(p.poll() is None for p in procs):   # progress line every 5 min in the Kaggle log
        time.sleep(5)
        if time.time() - last < 300:
            continue
        last = time.time()
        print(f"[{(time.time() - t) / 60:.0f} min] videos extracted: {len(glob.glob('data/*/crops/*/frames.json'))}"
              f" | feature files: {len(glob.glob('data/*/feats/*.npy'))}"
              f" | trained models: {len(glob.glob('runs/**/model.keras', recursive=True))}", flush=True)
    codes = [p.wait() for p in procs]
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)
    assert not any(codes), f"a command failed: {codes}"

## 1. Face crops for FF++ c23 (longest step, both GPUs)

In [ ]:
run(*[f'python code/prepare_data.py ffpp --root "{FF}" --splits-dir splits --out data/ffpp_c23 {Q} --shard {i}/2 '
      f'> extract{i}.log 2>&1' for i in range(2)])
!tail -n 2 extract0.log extract1.log

## 2. Cross-dataset and robustness test sets

In [ ]:
jobs = [f'python code/prepare_data.py ffpp --root "{FF}" --splits-dir splits --split test --subsample 3 '
        f'--perturb {p} --out data/ffpp_c23_{p.replace(":", "")} {Q}' for p in ("jpeg:30", "resize:0.25", "noise:20", "blur:3")]
if CDF:
    jobs.append(f'python code/prepare_data.py celebdf --root "{CDF}" --out data/celebdf {Q}')
if DFDC:
    jobs.append(f'python code/prepare_data.py dfdc --root "{DFDC}" --out data/dfdc {Q}')
run(" && ".join(jobs[0::2]) + " > tests0.log 2>&1", " && ".join(jobs[1::2]) + " > tests1.log 2>&1")
EXTRA = sorted(d for d in glob.glob("data/*") if d != "data/ffpp_c23")
print(EXTRA)

## 3. Temporal features, then the three cue models

In [ ]:
run(f"python code/temporal.py --features data/ffpp_c23 {' '.join(EXTRA)} > features.log 2>&1")
T = f"--set data/ffpp_c23 --fold full --train-frames 4 --predict {' '.join(EXTRA)}"
run(f"python code/train.py --cue spatial {T} > spatial.log 2>&1",
    f"(python code/train.py --cue frequency {T} && python code/train.py --cue temporal {T}) > freq_temporal.log 2>&1")
!tail -n 3 spatial.log freq_temporal.log

## 4. Fusion, baselines, tables

In [ ]:
SETS = "val test " + " ".join(os.path.basename(d) for d in EXTRA)
R = f"--run runs/ffpp_c23 --meta-source val --eval {SETS}"
run(f"python code/stacking.py {R} --export models")
for extra in ("--fusion mean", "--fusion wmean", "--fusion logreg",
              "--cues spatial,frequency", "--cues spatial,temporal", "--cues frequency,temporal"):
    run(f"python code/stacking.py {R} {extra}")
run(f"python code/stacking.py --run runs/ffpp_c23 --meta-source insample --eval {SETS}")
run("python code/evaluate.py --seeds 0 --fused stacking_valmeta --deletion 100")
run("zip -qr download.zip models results")
print(open("results/tables.md").read())